### **Part A**

**Q1**

[Embedded submission image omitted from public copy.]

**Q2**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv('hotel_bookings.csv')
selected_columns = ['hotel', 'lead_time', 'arrival_date_month', 'adults', 'children', 'babies', 'is_canceled', 'adr', 'total_of_special_requests', 'stays_in_week_nights', 'stays_in_weekend_nights']
df_selected = df[selected_columns]
df_sampled = df_selected.sample(n=10000, random_state=7)

selected_columns=['hotel', 'lead_time', 'is_canceled', 'adr']
df_sampled = df_sampled[selected_columns]
df_sampled_200 = df_sampled.sample(n=200, random_state=7)

**Q 3.A**

In [ ]:
numerical_columns_with_limits = {
    'adr': 1000,
    'lead_time': 1000,
}

for col, max_value in numerical_columns_with_limits.items():
    if col in ['adr', 'lead_time']:
        plt.figure(figsize=(8, 5))
        sns.histplot(df_sampled_200[df_sampled_200[col] <= max_value][col], kde=False, bins=15, color='blue')
        plt.title(f'Histogram of {col} ')
        plt.xlabel(col)
        plt.ylabel('Frequency')
        plt.show()

        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df_sampled_200[df_sampled_200[col] <= max_value][col], color='skyblue')
        plt.title(f'Box Plot of {col} ')
        plt.xlabel(col)
        plt.show()

categorical_columns = ['hotel', 'is_canceled']

for col in categorical_columns:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df_sampled_200, x=col, palette='pastel')
    plt.title(f'Bar Plot of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()

**Q 3.B**

**Q 3.C**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Scatter plot with regression line
plt.figure(figsize=(10, 6))
sns.regplot(x='adr', y='lead_time', data=df_sampled_200, scatter_kws={'alpha':0.6}, line_kws={"color":"red"})

# Adding labels and title
plt.ylabel('Lead Time')
plt.xlabel('ADR (Average Daily Rate)')
plt.title('ADR as a Function of Lead Time with Linear Regression Line')

# Displaying the plot
plt.show()

**Q 3.D**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


plt.figure(figsize=(10, 6))
sns.boxplot(x='is_canceled', y='lead_time',hue='is_canceled', data=df_sampled_200, palette='Set2')


plt.xlabel('Is Canceled')
plt.ylabel('Lead Time')
plt.title('Distribution of Lead Time by Cancellation Status')

# Displaying the plot
plt.show()

# Box plot for `lead_time` grouped by `is_canceled`
plt.figure(figsize=(10, 6))
sns.boxplot(x='hotel', y='lead_time',hue='hotel', data=df_sampled_200, palette='Set2')

# Adding labels and title
plt.xlabel('hotel')
plt.ylabel('Lead Time')
plt.title('Distribution of Lead Time by hotel')

# Displaying the plot
plt.show()




[Embedded submission image omitted from public copy.]

**Q4**

In [ ]:
import numpy as np
import pandas as pd
df_sampled_encoded_200 = df_sampled_200.copy()
# Map 'hotel' column to 0 and 1
df_sampled_encoded_200['hotel'] = df_sampled_200['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})

# Extracting features and target variable
X = df_sampled_encoded_200[['hotel', 'is_canceled', 'adr']].to_numpy()
y = df_sampled_encoded_200['lead_time'].to_numpy()

# Adding a column of 1s for the intercept
X = np.c_[np.ones(X.shape[0]), X]

# Compute beta_hat using the normal equation
beta_hat_200 = np.linalg.inv(X.T @ X) @ X.T @ y

# Display results
print(f"Beta Hat Coefficients: {beta_hat_200}")
print(f"Intercept (β₀): {beta_hat_200[0]}")
print(f"hotel Coefficient (β₁): {beta_hat_200[1]}")
print(f"is canceled Coefficient (β₂): {beta_hat_200[2]}")
print(f"adr Coefficient (β₃): {beta_hat_200[3]}")


[Embedded submission image omitted from public copy.]

**Q5**

In [ ]:
import statsmodels.api as sm
import pandas as pd
from scipy import stats

X = df_sampled_encoded_200[['hotel', 'is_canceled', 'adr']]
X = sm.add_constant(X)  # Add intercept
y = df_sampled_encoded_200['lead_time']


# Fit the model
model = sm.OLS(y, X).fit()
stderr_200 = model.bse.iloc[0]


# Manually calculate ANOVA components

# Total sum of squares
ss_total = ((y - y.mean())**2).sum()

# Residual sum of squares
ss_residual = sum(model.resid**2)

# Regression sum of squares
ss_regression = ss_total - ss_residual

# Degrees of freedom
df_total = len(y) - 1  # Total degrees of freedom
df_residual = len(y) - X.shape[1]  # Residual degrees of freedom
df_regression = X.shape[1] - 1  # Regression degrees of freedom (without constant)

# Mean squares
ms_regression = ss_regression / df_regression
ms_residual = ss_residual / df_residual

# F-statistic
f_statistic = ms_regression / ms_residual

# p-value for the F-statistic
p_value = 1 - stats.f.cdf(f_statistic, df_regression, df_residual)

# Display the ANOVA table manually
anova_table = pd.DataFrame({
    'Source': ['Regression', 'Residual', 'Total'],
    'SS': [ss_regression, ss_residual, ss_total],
    'df': [df_regression, df_residual, df_total],
    'MS': [ms_regression, ms_residual, None],
    'F': [f_statistic, None, None],
    'p-value': [p_value, None, None]
})

print(anova_table)

# Calculate R-squared (R²)
r_squared = ss_regression / ss_total

# Calculate Adjusted R-squared (R²_adj)
n = len(y)  # Number of observations
k = X.shape[1] - 1  # Number of predictors (excluding constant)

r_squared_adj = 1 - (ss_residual / ss_total) * ((n - 1) / (n - k - 1))

# Display the results
print(f"R-squared: {r_squared}")
print(f"Adjusted R-squared: {r_squared_adj}")



[Embedded submission image omitted from public copy.]

**Q6**

In [ ]:
se_beta = model.bse

df_residual = model.df_resid

alpha = 0.05
t_value = stats.t.ppf(1 - alpha / 2, df_residual)

confidence_intervals = pd.DataFrame({
    'Coefficient': model.params,
    'Std Error': se_beta,
    'Lower Bound': model.params - t_value * se_beta,
    'Upper Bound': model.params + t_value * se_beta
})

print(confidence_intervals)


calc original beta (using full data)

In [ ]:
import numpy as np
import pandas as pd
df_sampled_encoded_10000 = df_sampled.copy()
# Map 'hotel' column to 0 and 1
df_sampled_encoded_10000['hotel'] = df_sampled['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})

# Extracting features and target variable
X_10000 = df_sampled_encoded_10000[['hotel', 'is_canceled', 'adr']].to_numpy()
y_10000 = df_sampled_encoded_10000['lead_time'].to_numpy()

# Adding a column of 1s for the intercept
X_10000 = np.c_[np.ones(X_10000.shape[0]), X_10000]

# Compute beta_hat using the normal equation
beta_hat_10000 = np.linalg.inv(X_10000.T @ X_10000) @ X_10000.T @ y_10000

# Display results
print(f"Intercept (β₀): {beta_hat_10000[0]}")
print(f"hotel Coefficient (β₁): {beta_hat_10000[1]}")
print(f"is canceled Coefficient (β₂): {beta_hat_10000[2]}")
print(f"adr Coefficient (β₃): {beta_hat_10000[3]}")

[Embedded submission image omitted from public copy.]

**Q7**

In [ ]:
import numpy as np

# Calculate t-statistics for each coefficient
t_stats = beta_hat_200 / np.sqrt(np.diagonal(np.linalg.inv(X.T @ X) * np.var(y - X @ beta_hat_200)))

# Standard errors of the coefficients
standard_errors = np.sqrt(np.diagonal(np.linalg.inv(X.T @ X) * np.var(y - X @ beta_hat_200)))

# Calculate p-values (two-tailed test)
from scipy import stats
p_values = [2 * (1 - stats.t.cdf(np.abs(t_stat), df=len(y)-len(beta_hat_200))) for t_stat in t_stats]

# Print results
print("t-statistics:", t_stats)
print("Standard Errors:", standard_errors)
print("p-values:", p_values)

print("\nCoefficients and t-test results:")
for coef, t_val, p_val in zip(model.params.index, t_stats, p_values):
    print(f"{coef}: t-value = {t_val:.3f}, p-value = {p_val:.3g}")
    if p_val < 0.05:
        print(f"{coef} is significantly different from zero at the 95% significance level.\n")
    else:
        print(f"{coef} is not significantly different from zero at the 95% significance level.\n")


[Embedded submission image omitted from public copy.]

**Q8**

In [ ]:
residuals = y - model.fittedvalues

import matplotlib.pyplot as plt
plt.scatter(model.fittedvalues, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted values')
plt.show()


In [ ]:
# Histogram of residuals
plt.figure(figsize=(8, 6))
sns.histplot(residuals, kde=True)
plt.title('Histogram of Residuals')
plt.show()

# Q-Q plot
sm.qqplot(residuals, line='s')
plt.title('Q-Q Plot')
plt.show()


[Embedded submission image omitted from public copy.]

**Q9**

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm


df_sampled_encoded_9800_unique = df_sampled_encoded_10000[~df_sampled_encoded_10000.isin(df_sampled_encoded_200)].dropna()
df_sampled_encoded_1000 = df_sampled_encoded_9800_unique.sample(n=1000, random_state=7)

X_new = df_sampled_encoded_1000[['hotel', 'is_canceled', 'adr']]
y_new = df_sampled_encoded_1000['lead_time']
X_new = sm.add_constant(X_new)
y_hat = beta_hat_200[0] + beta_hat_200[1]*X_new['hotel'] + beta_hat_200[2]*X_new['is_canceled'] + beta_hat_200[3]*X_new['adr']

n = len(X_new)
p = X_new.shape[1]

X_new_matrix = X_new.values
C = np.linalg.inv(X_new_matrix.T @ X_new_matrix)  # (X_new^T X_new)^-1

# Calculate the residual variance (sigma^2)
residuals = y_new - y_hat
sigma_squared = np.var(residuals)

# Extract diagonal elements from (X_new_matrix @ C @ X_new_matrix.T)
se_y_hat = np.sqrt(sigma_squared * (1 + np.diagonal(X_new_matrix @ C @ X_new_matrix.T)))

# 2. Confidence interval for the predictions using normal distribution
alpha = 0.05
z_value = norm.ppf(1 - alpha/2)

# Calculate the margin of error for each predicted value
margin_of_error = z_value * se_y_hat

# Calculate the lower and upper bounds of the confidence interval
lower_bound = y_hat - margin_of_error
upper_bound = y_hat + margin_of_error

# Initialize the count variable for the number of values within the confidence interval
count_within_interval = 0

# Iterate through the values of y_new and check if they fall within the corresponding confidence intervals
for val, lower, upper in zip(y_new, lower_bound, upper_bound):
    if lower <= val <= upper:  # Check if the value is within the confidence interval
        count_within_interval += 1

# Calculate the percentage of values within the confidence interval
percentage_within_interval = (count_within_interval / len(y_hat)) * 100

# Print the result
print(f"Percentage of values within the confidence interval: {percentage_within_interval:.2f}%")


[Embedded submission image omitted from public copy.]

**Q10**

In [ ]:
df_sampled_encoded_200_z = df_sampled_encoded_200.copy()
df_sampled_encoded_200_z['Z12'] = df_sampled_encoded_200['hotel'] * df_sampled_encoded_200['is_canceled']
df_sampled_encoded_200_z['Z13'] = df_sampled_encoded_200_z['hotel'] * df_sampled_encoded_200_z['adr']
df_sampled_encoded_200_z['Z23'] = df_sampled_encoded_200_z['is_canceled'] * df_sampled_encoded_200_z['adr']

X_new = df_sampled_encoded_200_z[['hotel', 'is_canceled', 'adr', 'Z12', 'Z13', 'Z23']]

X_new = sm.add_constant(X_new)

y_new = df_sampled_encoded_200_z['lead_time']

model = sm.OLS(y_new, X_new).fit()

print(model.params)


In [ ]:
import statsmodels.api as sm

# Define all possible combinations of the predictors
models = {
    'Original Model ': ['hotel', 'is_canceled', 'adr'],  # Basic model
    'Interactions Model ': ['hotel', 'is_canceled', 'adr', 'Z12', 'Z13', 'Z23'],
}

# Initialize a dictionary to store model metrics
model_metrics = {}

# Try each model and evaluate it
for model_name, predictors in models.items():
    X = df_sampled_encoded_200_z[predictors]
    X = sm.add_constant(X)
    y = df_sampled_encoded_200_z['lead_time']

    # Fit the model
    model = sm.OLS(y, X).fit()

    # Get AIC, BIC, and calculate the training error (Rtr)
    aic = model.aic
    bic = model.bic



    # Store the metrics for the model
    model_metrics[model_name] = {
        'AIC': aic,
        'BIC': bic,

        }

# Sort models based on AIC, BIC, and Rtr (Training Error)
sorted_by_aic = sorted(model_metrics.items(), key=lambda x: x[1]['AIC'])
sorted_by_bic = sorted(model_metrics.items(), key=lambda x: x[1]['BIC'])


print("models based on AIC:")
for model_name, metrics in sorted_by_aic:
    print(f"{model_name}: AIC={metrics['AIC']}")

print("\nmodels based on BIC:")
for model_name, metrics in sorted_by_bic:
    print(f"{model_name}: BIC={metrics['BIC']}")



[Embedded submission image omitted from public copy.]

### **Part B**

**Q1**

[Embedded submission image omitted from public copy.]

**Q2**

In [ ]:
import numpy as np
import pandas as pd

df_sampled_encoded_200 = df_sampled_200.copy()
# Map 'hotel' column to 0 and 1
df_sampled_encoded_200['hotel'] = df_sampled_200['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})

# Extracting features and target variable
X_2 = df_sampled_encoded_200[['hotel', 'lead_time', 'adr']].to_numpy()
y_2 = df_sampled_encoded_200['is_canceled'].to_numpy()

**Q3**

אנחנו משתמשים באותן משתנים ובאותו מדגם כמו בשאלה הקודמת, ולכן הגרף וההסברים נשארים ללא שינוי

**Q4**

In [ ]:
import statsmodels.api as sm

# Add a constant term for the intercept
X_2_c = sm.add_constant(X_2)

# Fit logistic regression model
logit_model = sm.Logit(y_2, X_2_c)
result = logit_model.fit()

# Display the results
print(result.summary())


[Embedded submission image omitted from public copy.]

**Q5**

In [ ]:
# Extract confidence intervals for the coefficients from the sampled data
conf_intervals_sampled_200 = result.conf_int()
print("Confidence intervals for sampled data:")
print(conf_intervals_sampled_200)

# Create a dictionary for confidence intervals
conf_intervals_dict = {
    'const': conf_intervals_sampled_200[0],
    'hotel': conf_intervals_sampled_200[1],
    'lead_time': conf_intervals_sampled_200[2],
    'adr': conf_intervals_sampled_200[3]
}

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Assuming df_sampled is already defined
df_sampled_encoded_10000 = df_sampled.copy()
# Map 'hotel' column to 0 and 1
df_sampled_encoded_10000['hotel'] = df_sampled['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})

# Extracting features and target variable
X_10000_2 = df_sampled_encoded_10000[['hotel', 'lead_time', 'adr']].to_numpy()
y_10000_2 = df_sampled_encoded_10000['is_canceled'].to_numpy()

# Adding a column of 1s for the intercept
X_10000_2 = np.c_[np.ones(X_10000_2.shape[0]), X_10000_2]

# Add a constant term for the intercept
X_10000_2 = sm.add_constant(X_10000_2)

# Fit logistic regression model
logit_model = sm.Logit(y_10000_2, X_10000_2)
result = logit_model.fit(disp=0)

# List of parameter names
param_names = ['const', 'hotel', 'lead_time', 'adr']

for param_name, param_value in zip(param_names, result.params):
    lower, upper = conf_intervals_dict[param_name]
    is_within = lower <= param_value <= upper
    print(f"{param_name}: {param_value:.4f} (CI: [{lower:.4f}, {upper:.4f}]) - Within Interval: {is_within}")


[Embedded submission image omitted from public copy.]

**Q6**

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Adding interaction terms
Z12 = X_2[:, 0] * X_2[:, 1]  # X1 * X2
Z13 = X_2[:, 0] * X_2[:, 2]  # X1 * X3
Z23 = X_2[:, 1] * X_2[:, 2]  # X2 * X3

# Combine the original features with the interaction terms
X_2_interactions = np.c_[X_2, Z12, Z13, Z23]

# Add a constant term for the intercept
X_2_interactions = sm.add_constant(X_2_interactions)

# Fit logistic regression model with interaction terms
logit_model_interactions = sm.Logit(y_2, X_2_interactions)
result_interactions = logit_model_interactions.fit(disp=0)

# List of parameter names (including interaction terms)
param_names = ['const', 'hotel', 'lead_time', 'adr', 'hotel*lead_time', 'hotel*adr', 'lead_time*adr']

# Print the results with formatting
print("\nLogistic Regression Coefficients with Interaction Terms:")
print("-" * 60)
print(f"{'Parameter':<25}{'Coefficient':<15}")
print("-" * 60)
for param_name, coef in zip(param_names, result_interactions.params):
    print(f"{param_name:<25}{coef:.4f}")
print("-" * 60)


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2

# Base model (without interaction terms)
logit_model_base = sm.Logit(y_2, X_2)
result_base = logit_model_base.fit(disp=0)

# Combine the original features with the interaction terms
X_2_interactions = np.c_[X_2, Z12, Z13, Z23]
X_2_interactions = sm.add_constant(X_2_interactions)

logit_model_interactions = sm.Logit(y_2, X_2_interactions)
result_interactions = logit_model_interactions.fit(disp=0)

# Compute log-likelihood for both models
log_likelihood_base = result_base.llf
log_likelihood_interactions = result_interactions.llf

# Compute Likelihood Ratio Statistic
lr_stat = 2 * (log_likelihood_interactions - log_likelihood_base)

# Compute degrees of freedom (difference in the number of parameters)
df_diff = len(result_interactions.params) - len(result_base.params)

# Compute p-value
p_value = 1 - chi2.cdf(lr_stat, df_diff)

# Display the result
print(f"Likelihood Ratio Statistic: {lr_stat:.4f}")
print(f"Degrees of Freedom: {df_diff}")
print(f"P-value: {p_value:.4f}")

# Decision based on p-value
if p_value < 0.05:
    print("The model with interaction terms is preferred (the difference is significant).")
else:
    print("The model without interaction terms is preferred (the difference is not significant).")


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Base model (without interaction terms)
logit_model_base = sm.Logit(y_2, X_2)
result_base = logit_model_base.fit(disp=0)

# Combine the original features with the interaction terms
X_2_interactions = np.c_[X_2, Z12, Z13, Z23]
X_2_interactions = sm.add_constant(X_2_interactions)

logit_model_interactions = sm.Logit(y_2, X_2_interactions)
result_interactions = logit_model_interactions.fit(disp=0)

# Number of observations
n = X_2.shape[0]

# Log-likelihood for both models
log_likelihood_base = result_base.llf
log_likelihood_interactions = result_interactions.llf

# Number of parameters in each model
k_base = len(result_base.params)
k_interactions = len(result_interactions.params)

# Calculate AIC and BIC for the base model
AIC_base = 2 * k_base - 2 * log_likelihood_base
BIC_base = np.log(n) * k_base - 2 * log_likelihood_base

# Calculate AIC and BIC for the model with interaction terms
AIC_interactions = 2 * k_interactions - 2 * log_likelihood_interactions
BIC_interactions = np.log(n) * k_interactions - 2 * log_likelihood_interactions

# Display results
print(f"Base Model AIC: {AIC_base:.4f}")
print(f"Base Model BIC: {BIC_base:.4f}")
print(f"Model with Interaction Terms AIC: {AIC_interactions:.4f}")
print(f"Model with Interaction Terms BIC: {BIC_interactions:.4f}")

[Embedded submission image omitted from public copy.]

Q7

In [ ]:
selected_columns = ['hotel', 'lead_time', 'is_canceled', 'adr','adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations']
df_selected = df[selected_columns]
df_sampled_200_big = df_selected.sample(n=200, random_state=7)

df_sampled_encoded_200_big = df_sampled_200_big.copy()

# Map 'hotel' column to 0 and 1
df_sampled_encoded_200_big['hotel'] = df_sampled_200_big['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})

# Extracting features and target variable
X_big = df_sampled_encoded_200_big[['hotel', 'lead_time', 'adr','adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations']].to_numpy()
y_big = df_sampled_encoded_200_big['is_canceled'].to_numpy()


import itertools
import numpy as np

# Get all pairs of column indices (i, j) where i < j
pairs = list(itertools.combinations(range(X_big.shape[1]), 2))

# Create an empty list to store the interaction terms
interaction_terms = []

# Calculate interaction terms for each pair of columns
for (i, j) in pairs:
    interaction_terms.append(X_big[:, i] * X_big[:, j])

# Stack the interaction terms with the original X_big matrix
X_big_interactions = np.column_stack([X_big] + interaction_terms)

# Define the target variable
y_big = df_sampled_encoded_200_big['is_canceled'].to_numpy()

# Now X_big_interactions contains the original features plus the interaction terms

print (X_big_interactions.shape)


In [ ]:
import statsmodels.api as sm
import numpy as np

# Function to calculate AIC for a given set of predictors
def calculate_aic(X, y):
    X_with_intercept = sm.add_constant(X)  # Add an intercept term
    model = sm.OLS(y, X_with_intercept).fit()
    return model.aic, model

# Forward stepwise selection based on AIC
def forward_stepwise_aic(X, y):
    n_features = X.shape[1]
    selected_features = []
    remaining_features = list(range(n_features))
    current_aic = np.inf
    best_model = None

    while remaining_features:
        aic_with_candidates = []

        for feature in remaining_features:
            # Test the current feature by adding it to the selected set
            candidate_features = selected_features + [feature]
            X_candidate = X[:, candidate_features]
            aic, model = calculate_aic(X_candidate, y)
            aic_with_candidates.append((aic, feature, model))

        # Find the feature that minimizes AIC
        aic_with_candidates.sort()
        best_aic, best_feature, model = aic_with_candidates[0]

        if best_aic < current_aic:
            current_aic = best_aic
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
            best_model = model
        else:
            # No improvement in AIC, stop the process
            break

    return selected_features, best_model

# Perform forward stepwise regression on the dataset
selected_features, final_model = forward_stepwise_aic(X_big_interactions, y_big)

# Create feature names for original features and interaction terms
original_feature_names = ['hotel', 'lead_time', 'adr', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations']

# Generate names for interaction terms
interaction_feature_names = [
    f"{original_feature_names[i]} * {original_feature_names[j]}"
    for i, j in pairs
]

# Combine original and interaction feature names
all_feature_names = original_feature_names + interaction_feature_names

# Map indices to feature names
selected_feature_names = [all_feature_names[i] for i in selected_features]

# Output the selected feature names and final model summary
print("Selected Features:", selected_feature_names)



In [ ]:
import statsmodels.api as sm
import numpy as np

# Function to calculate AIC for a given set of predictors
def calculate_aic(X, y):
    X_with_intercept = sm.add_constant(X)  # Add an intercept term
    model = sm.OLS(y, X_with_intercept).fit()
    return model.aic, model

# Backward stepwise selection based on AIC
def backward_stepwise_aic(X, y):
    n_features = X.shape[1]
    selected_features = list(range(n_features))  # Start with all features
    current_aic, best_model = calculate_aic(X, y)

    while len(selected_features) > 0:
        aic_with_candidates = []

        for feature in selected_features:
            # Test removing the current feature
            candidate_features = [f for f in selected_features if f != feature]
            X_candidate = X[:, candidate_features]
            aic, model = calculate_aic(X_candidate, y)
            aic_with_candidates.append((aic, feature, model))

        # Find the feature whose removal minimizes AIC
        aic_with_candidates.sort()
        best_aic, removed_feature, model = aic_with_candidates[0]

        if best_aic < current_aic:
            current_aic = best_aic
            selected_features.remove(removed_feature)
            best_model = model
        else:
            # No improvement in AIC, stop the process
            break

    return selected_features, best_model

# Perform backward stepwise regression on the dataset
selected_features, final_model = backward_stepwise_aic(X_big_interactions, y_big)

# Map feature indices to names
original_feature_names = ['hotel', 'lead_time', 'adr', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations']
interaction_feature_names = [
    f"{original_feature_names[i]} * {original_feature_names[j]}"
    for i, j in pairs
]
all_feature_names = original_feature_names + interaction_feature_names
selected_feature_names = [all_feature_names[i] for i in selected_features]

# Output the selected feature names and final model summary
print("Selected Features:", selected_feature_names)


[Embedded submission image omitted from public copy.]

Q8

In [ ]:
import statsmodels.api as sm
import numpy as np

# Function to calculate BIC for a given set of predictors
def calculate_bic(X, y):
    X_with_intercept = sm.add_constant(X)  # Add an intercept term
    model = sm.OLS(y, X_with_intercept).fit()
    return model.bic, model

# Forward stepwise selection based on BIC
def forward_stepwise_bic(X, y):
    n_features = X.shape[1]
    selected_features = []
    remaining_features = list(range(n_features))
    current_bic = np.inf
    best_model = None

    while remaining_features:
        bic_with_candidates = []

        for feature in remaining_features:
            # Test the current feature by adding it to the selected set
            candidate_features = selected_features + [feature]
            X_candidate = X[:, candidate_features]
            bic, model = calculate_bic(X_candidate, y)
            bic_with_candidates.append((bic, feature, model))

        # Find the feature that minimizes BIC
        bic_with_candidates.sort()
        best_bic, best_feature, model = bic_with_candidates[0]

        if best_bic < current_bic:
            current_bic = best_bic
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
            best_model = model
        else:
            # No improvement in BIC, stop the process
            break

    return selected_features, best_model

# Perform forward stepwise regression on the dataset
selected_features, final_model = forward_stepwise_bic(X_big_interactions, y_big)

# Map feature indices to names
original_feature_names = ['hotel', 'lead_time', 'adr', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations']
interaction_feature_names = [
    f"{original_feature_names[i]} * {original_feature_names[j]}"
    for i, j in pairs
]
all_feature_names = original_feature_names + interaction_feature_names
selected_feature_names = [all_feature_names[i] for i in selected_features]

# Output the selected feature names and final model summary
print("Selected Features:", selected_feature_names)


In [ ]:
import statsmodels.api as sm
import numpy as np

# Function to calculate BIC for a given set of predictors
def calculate_bic(X, y):
    X_with_intercept = sm.add_constant(X)  # Add an intercept term
    model = sm.OLS(y, X_with_intercept).fit()
    return model.bic, model

# Backward stepwise selection based on BIC
def backward_stepwise_bic(X, y):
    n_features = X.shape[1]
    selected_features = list(range(n_features))  # Start with all features
    current_bic, best_model = calculate_bic(X, y)

    while len(selected_features) > 0:
        bic_with_candidates = []

        for feature in selected_features:
            # Test removing the current feature
            candidate_features = [f for f in selected_features if f != feature]
            X_candidate = X[:, candidate_features]
            bic, model = calculate_bic(X_candidate, y)
            bic_with_candidates.append((bic, feature, model))

        # Find the feature whose removal minimizes BIC
        bic_with_candidates.sort()
        best_bic, removed_feature, model = bic_with_candidates[0]

        if best_bic < current_bic:
            current_bic = best_bic
            selected_features.remove(removed_feature)
            best_model = model
        else:
            # No improvement in BIC, stop the process
            break

    return selected_features, best_model

# Perform backward stepwise regression on the dataset
selected_features, final_model = backward_stepwise_bic(X_big_interactions, y_big)

# Map feature indices to names
original_feature_names = ['hotel', 'lead_time', 'adr', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations']
interaction_feature_names = [
    f"{original_feature_names[i]} * {original_feature_names[j]}"
    for i, j in pairs
]
all_feature_names = original_feature_names + interaction_feature_names
selected_feature_names = [all_feature_names[i] for i in selected_features]

# Output the selected feature names and final model summary
print("Selected Features:", selected_feature_names)


[Embedded submission image omitted from public copy.]